# ESP-Lab drift figure gallery

Discover all figures generated by the drift and initialization suite (`jupyter/s2d_drift/0_*` through `2b_*`, plus the monthly/daily drift runners) and build a searchable, responsive webpage. Figures are read from the drift figure root (`esp_lab.env_paths.drift_figure_root()`, override with `ESP_LAB_DRIFT_FIGURE_ROOT`) and grouped as **Lead-time Drift** (`fig_drift_*`) and **Initial-condition Audit** (`fig_ic_*`).


In [ ]:
# Imports
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "esp_lab").is_dir()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the ESP-Lab checkout.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from esp_lab import env_paths
from esp_lab.diagnostics.web import (
    discover_workflow_figures,
    generate_diagnostics_webpage,
)


In [ ]:
# ============================================================
# Configuration
# ============================================================
DIAG_DIR = env_paths.drift_figure_root()
FIGURE_PATTERN = "fig_*"
INCLUDE_SUBDIRS = True
PORTAL_URL = "https://portal.nersc.gov/cfs/e3sm/zhan391/e3sm-s2d_dev/index.html"

## 1. Discover workflow figures

Scan the directory rather than trusting old manifest entries. The synchronized catalog below is exactly what the webpage will show.

In [ ]:
manifest = discover_workflow_figures(
    DIAG_DIR,
    pattern=FIGURE_PATTERN,
    write_manifest=True,
    include_subdirs=INCLUDE_SUBDIRS,
)
catalog = pd.DataFrame(manifest["figures"])[
    ["file", "group", "mode", "metric", "title", "caption"]
]
print(f"Found {len(catalog)} workflow figures in {DIAG_DIR}")
display(catalog.groupby("group").size().rename("figures").to_frame())
display(catalog)

## 2. Build the webpage

In [ ]:
output_html = generate_diagnostics_webpage(
    DIAG_DIR,
    discover_figures=True,
    figure_pattern=FIGURE_PATTERN,
    include_subdirs=INCLUDE_SUBDIRS,
)
print(f"Webpage created with {len(catalog)} figures: {output_html}")
display(HTML(f'<a href="{PORTAL_URL}" target="_blank">Open the ESP-Lab drift figure gallery</a>'))

The resulting `index.html` provides group and metric filters, text search, responsive figure cards, full-resolution viewing, and figure comparison. Generation also makes the cataloged figures publicly readable and the gallery directory publicly traversable so the portal can serve every image. Re-run the two sections whenever workflow figures are added, replaced, or removed.